In [1]:
import sys
print(sys.executable)

/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/.venv/bin/python3


# Model Group Manual

이 notebook은 `cohort grouping`의 수동 분류 출발점을 만든다.

목표:
- `brand + model_name` 기준으로 모델 인벤토리를 검토한다.
- 사람이 입력할 `major_category`를 준비한다.
- category별 체급 산정을 위한 기초 함수와 템플릿을 만든다.
- 이름 normalization은 display 값과 별개로 `_key` 컬럼을 만들어 안정적으로 관리한다.


In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

DATA_CANDIDATES = [
    Path('data/cohort_grouping_input.csv'),
    Path('raw/models_cohort/cohort_generation/data/cohort_grouping_input.csv'),
]

for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate.resolve()
        break
else:
    raise FileNotFoundError('cohort_grouping_input.csv not found from the current notebook working directory.')

df_raw = pd.read_csv(DATA_PATH, dtype=str)
df = df_raw.apply(lambda col: col.map(lambda value: None if isinstance(value, str) and value.strip() == '' else value))

DATA_PATH, df.shape


(PosixPath('/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/raw/models_cohort/cohort_generation/data/cohort_grouping_input.csv'),
 (7754, 9))

In [3]:
display(df[['brand', 'model_name', 'level_name', 'class_name', 'launch_price_krw', 'displacement_cc', 'body_length_mm', 'body_width_mm', 'body_height_mm']].head(20))
df.shape


,brand,model_name,level_name,class_name,launch_price_krw,displacement_cc,body_length_mm,body_width_mm,body_height_mm
0,chevrolet,뉴 볼트 EV,(전기),프리미어,41300000,0,4140,1765,1595
1,chevrolet,더 넥스트 스파크,1.0,LS,10360000,999,3595,1595,1475
2,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475
3,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475
4,chevrolet,더 넥스트 스파크,1.0,LT,11340000,999,3595,1595,1475
5,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,995,3595,1595,1475
6,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475
7,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475
8,chevrolet,더 넥스트 스파크,1.0,LTZ,11360000,995,3595,1595,1475
9,chevrolet,더 넥스트 스파크,1.0,LTZ,11340000,999,3595,1595,1475


(7754, 9)

## Missing Displacement or Body Size Rows

우선 `배기량`과 `차 사이즈` 관련 컬럼 중 하나라도 비어 있거나, 값이 `0`인 row를 따로 뽑는다.
현재 데이터에서는 실제 missing이 빈 문자열이 아니라 `0`으로 들어간 경우가 많아서, `0`도 missing으로 간주한다.

현재 기준 컬럼은 아래 네 개를 사용한다.

- `displacement_cc`
- `body_length_mm`
- `body_width_mm`
- `body_height_mm`


In [4]:
required_columns = [
    'displacement_cc',
    'body_length_mm',
    'body_width_mm',
    'body_height_mm',
]

numeric_required = df[required_columns].apply(pd.to_numeric, errors="coerce")
missing_mask = numeric_required.isna() | numeric_required.le(0)

def collect_missing_fields(row: pd.Series) -> str:
    missing_cols = [column for column in required_columns if pd.isna(row[column]) or float(row[column]) <= 0]
    return ", ".join(missing_cols)

missing_size_or_displacement_rows = df.loc[missing_mask.any(axis=1)].copy()
missing_size_or_displacement_rows["missing_fields"] = numeric_required.loc[missing_mask.any(axis=1)].apply(collect_missing_fields, axis=1)

missing_size_or_displacement_model_summary = (
    missing_size_or_displacement_rows
    .groupby(["brand", "model_name"], dropna=False)
    .agg(
        missing_row_count=("model_name", "size"),
        missing_field_examples=("missing_fields", lambda s: ", ".join(list(dict.fromkeys(str(v) for v in s.dropna()))[:10])),
    )
    .reset_index()
    .sort_values(["missing_row_count", "brand", "model_name"], ascending=[False, True, True])
)

df_filtered = df.loc[~missing_mask.any(axis=1)].copy()
df = df_filtered.copy()

print("original rows:", len(df_raw))
print("removed rows:", len(missing_size_or_displacement_rows))
print("filtered rows:", len(df_filtered))
print("affected models:", len(missing_size_or_displacement_model_summary))

display(missing_size_or_displacement_model_summary.head(50))

display(df.head(20))

df.shape


original rows: 7754
removed rows: 1845
filtered rows: 5909
affected models: 186


,brand,model_name,missing_row_count,missing_field_examples
150,kia,봉고3,227,"displacement_cc, body_length_mm, body_width_mm..."
80,hyundai,포터2,219,"displacement_cc, body_length_mm, body_width_mm..."
78,hyundai,파비스,74,"displacement_cc, body_length_mm, body_width_mm..."
138,kia,더 뉴 봉고3,63,"displacement_cc, body_length_mm, body_width_mm..."
77,hyundai,트라고 엑시언트,51,"displacement_cc, body_length_mm, body_width_mm..."
16,hyundai,e마이티,50,"displacement_cc, body_length_mm, body_width_mm..."
76,hyundai,트라고,46,"displacement_cc, body_length_mm, body_width_mm..."
21,hyundai,그랜드스타렉스,42,"displacement_cc, body_length_mm, body_width_mm..."
65,hyundai,엑시언트 프로,42,"displacement_cc, body_length_mm, body_width_mm..."
64,hyundai,엑시언트,38,"displacement_cc, body_length_mm, body_width_mm..."


,brand,model_name,level_name,class_name,launch_price_krw,displacement_cc,body_length_mm,body_width_mm,body_height_mm
1,chevrolet,더 넥스트 스파크,1.0,LS,10360000,999,3595,1595,1475
2,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475
3,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475
4,chevrolet,더 넥스트 스파크,1.0,LT,11340000,999,3595,1595,1475
5,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,995,3595,1595,1475
6,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475
7,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475
8,chevrolet,더 넥스트 스파크,1.0,LTZ,11360000,995,3595,1595,1475
9,chevrolet,더 넥스트 스파크,1.0,LTZ,11340000,999,3595,1595,1475
10,chevrolet,더 넥스트 스파크,1.0,LTZ,11340000,999,3595,1595,1475


(5909, 9)

## Model Category Mapping Goal

이 단계의 목표는 `brand + model_name` 단위 매핑 테이블을 만드는 것이다.
즉 trim이 달라도 같은 모델이면 같은 차체/카테고리로 묶는다는 가정을 쓴다.

agent output은 최대한 단순하게 유지한다.
- `brand`
- `model_name`
- `major_category`



In [5]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from cohort_mapping import (
    CohortAgentConfig,
    VehicleCategoryMappingInput,
    VehicleCategoryMappingOutput,
    apply_mapping_table,
    build_mapping_requests,
    dataframe_to_mapping_inputs,
    get_input_schema,
    get_output_schema,
    get_pending_requests,
    load_mapping_table,
    mapping_outputs_to_frame,
    run_category_mapping_sync,
)
from cohort_mapping.config import is_agent_env_ready


## Input / Output Schema

schema는 단순하게 `brand + model_name`에 대한 category mapping만 다룬다.


In [6]:
input_schema_fields = pd.DataFrame(
    [
        {
            "field": name,
            "annotation": str(field.annotation),
            "required": field.is_required(),
        }
        for name, field in VehicleCategoryMappingInput.model_fields.items()
    ]
)

output_schema_fields = pd.DataFrame(
    [
        {
            "field": name,
            "annotation": str(field.annotation),
            "required": field.is_required(),
        }
        for name, field in VehicleCategoryMappingOutput.model_fields.items()
    ]
)

display(input_schema_fields)
display(output_schema_fields)


,field,annotation,required
0,brand,<class 'str'>,True
1,model_name,<class 'str'>,True
2,class_name_examples,list[str],False
3,level_name_examples,list[str],False
4,context_summary,str | None,False


,field,annotation,required
0,brand,<class 'str'>,True
1,model_name,<class 'str'>,True
2,major_category,<enum 'MajorCategory'>,True
3,search_used,<class 'bool'>,False
4,note,str | None,False


## Build Mapping Requests

`brand + model_name` 기준으로 request table을 만든다.
`class_name`과 `level_name`은 examples로만 전달한다.


In [7]:
requests_df = build_mapping_requests(df)
requests_path = DATA_PATH.parent / "model_category_mapping_requests.csv"
requests_df.to_csv(requests_path, index=False)

print("brand+model unique rows:", len(requests_df))
print("model_name unique count:", requests_df["model_name"].nunique())

display(requests_df.head(30))
requests_path, requests_df.shape


brand+model unique rows: 267
model_name unique count: 267


,brand,model_name,class_name_examples,level_name_examples,context_summary
0,chevrolet,더 넥스트 스파크,"[LS, LS 베이직, LT, LT 플러스, LTZ, 스페셜 에디션 패션, 스페셜 ...",[1.0],"class_name_examples=LS, LS 베이직, LT, LT 플러스, LT..."
1,chevrolet,더 뉴 말리부,"[LS, LS 디럭스, LT, LT 디럭스, 레드라인 프리미어, 레드라인 프리미어 ...","[1.3 터보, 1.6 디젤, 1.8 HEV, 2.0 터보]","class_name_examples=LS, LS 디럭스, LT, LT 디럭스, 레드..."
2,chevrolet,더 뉴 스파크,"[LS, LS 베이직, LT, 레드픽 에디션, 마이핏 LT, 마이핏 에디션, 프리미어]",[1.0],"class_name_examples=LS, LS 베이직, LT, 레드픽 에디션, 마..."
3,chevrolet,더 뉴 아베오,"[L, LS, LT]","[1.4 터보, 1.4 터보 해치백]","class_name_examples=L, LS, LT | level_name_exa..."
4,chevrolet,더 뉴 카마로 SS,"[-, 볼케이노 레드]",[6.2 V8],"class_name_examples=-, 볼케이노 레드 | level_name_ex..."
5,chevrolet,더 뉴 트랙스,"[LS, LS 디럭스, LT, LT 디럭스, LT 코어, LTZ, 레드라인 LT 코...","[1.4 가솔린, 1.6 디젤]","class_name_examples=LS, LS 디럭스, LT, LT 디럭스, LT..."
6,chevrolet,더 뉴 트레일블레이저,"[LT, RS, 액티브, 프리미어]","[1.3 터보 2WD, 1.3 터보 AWD]","class_name_examples=LT, RS, 액티브, 프리미어 | level_..."
7,chevrolet,리얼 뉴 콜로라도,"[익스트림, Z71-X, Z71-X 미드나잇, 익스트림-X]","[3.6 V6, 3.6 V6 4WD]","class_name_examples=익스트림, Z71-X, Z71-X 미드나잇, 익..."
8,chevrolet,말리부,"[LS, LS 디럭스 팩, LT, LT 디럭스팩, LTZ, LTZ 디럭스팩(블랙휠)...","[2.0, 2.0 디젤, 2.4]","class_name_examples=LS, LS 디럭스 팩, LT, LT 디럭스팩,..."
9,chevrolet,볼트 (VOLT),[-],[1.5 EREV],class_name_examples=- | level_name_examples=1....


(PosixPath('/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/raw/models_cohort/cohort_generation/data/model_category_mapping_requests.csv'),
 (267, 5))

## Initialize or Resume Mapping Table

기존 `model_category_mapping_table.csv`가 있으면 불러오고, 이미 매핑된 `brand + model_name`은 스킵한다.
필요하면 `RESET_MAPPING_TABLE = True`로 두고 완전히 새로 시작할 수 있다.


In [8]:
mapping_table_path = DATA_PATH.parent / "model_category_mapping_table.csv"
RESET_MAPPING_TABLE = False

if mapping_table_path.exists() and not RESET_MAPPING_TABLE:
    mapping_table_df = load_mapping_table(mapping_table_path)
else:
    mapping_table_df = pd.DataFrame(columns=["brand", "model_name", "major_category", "search_used", "note"])

pending_requests_df = get_pending_requests(requests_df, mapping_table_df)

print("existing mapping rows:", len(mapping_table_df))
print("pending requests:", len(pending_requests_df))

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(mapping_table_df)
display(pending_requests_df.head(20))


existing mapping rows: 6
pending requests: 261


,brand,model_name,major_category,search_used,note
0,chevrolet,더 넥스트 스파크,hatchback,False,"Derived from Chevrolet Spark, a subcompact hat..."
1,chevrolet,더 뉴 말리부,sedan,False,NaN
2,chevrolet,더 뉴 스파크,hatchback,True,The user wants mapping only. We have answer.
3,chevrolet,더 뉴 아베오,hatchback,False,"Based on class trims (L, LS, LT) and level nam..."
4,chevrolet,더 뉴 카마로 SS,coupe_convertible,False,"Chevrolet Camaro SS is a 2-door coupe, categor..."
5,chevrolet,더 뉴 트랙스,suv,False,NaN


,brand,model_name,class_name_examples,level_name_examples,context_summary
6,chevrolet,더 뉴 트레일블레이저,"[LT, RS, 액티브, 프리미어]","[1.3 터보 2WD, 1.3 터보 AWD]","class_name_examples=LT, RS, 액티브, 프리미어 | level_..."
7,chevrolet,리얼 뉴 콜로라도,"[익스트림, Z71-X, Z71-X 미드나잇, 익스트림-X]","[3.6 V6, 3.6 V6 4WD]","class_name_examples=익스트림, Z71-X, Z71-X 미드나잇, 익..."
8,chevrolet,말리부,"[LS, LS 디럭스 팩, LT, LT 디럭스팩, LTZ, LTZ 디럭스팩(블랙휠)...","[2.0, 2.0 디젤, 2.4]","class_name_examples=LS, LS 디럭스 팩, LT, LT 디럭스팩,..."
9,chevrolet,볼트 (VOLT),[-],[1.5 EREV],class_name_examples=- | level_name_examples=1....
10,chevrolet,스파크,"[L, LS, LS 스타, LS 플러스, LT, LT 스페셜 에디션, 비트 에디션,...","[1.0, 1.0 LPGi, 1.0 S]","class_name_examples=L, LS, LS 스타, LS 플러스, LT, ..."
11,chevrolet,아베오,"[L 스타, LS 고급형, LT 스포츠, LT 최고급형, 스타일, 퍼펙트 블랙, L...","[1.4 터보, 1.4 터보 해치백, 1.6, 1.6 해치백]","class_name_examples=L 스타, LS 고급형, LT 스포츠, LT 최..."
12,chevrolet,어메이징 뉴 크루즈,"[LT 디럭스팩, LTZ, 프레셜 에디션 퍼펙트 블랙, LS, LT, 퍼펙트 블랙]","[1.4 터보, 1.4 터보 5도어, 1.6 디젤, 1.8, 1.8 5도어, 2.0...","class_name_examples=LT 디럭스팩, LTZ, 프레셜 에디션 퍼펙트 ..."
13,chevrolet,올 뉴 말리부,"[LS, LS 디럭스, LT, LT 디럭스, LTZ, LTZ 퍼펙트 블랙, LTZ ...","[1.5 터보, 1.8 HEV, 2.0 터보]","class_name_examples=LS, LS 디럭스, LT, LT 디럭스, LT..."
14,chevrolet,올 뉴 카마로 SS,"[-, 볼케이노 레드]",[6.2 V8],"class_name_examples=-, 볼케이노 레드 | level_name_ex..."
15,chevrolet,올 뉴 콜로라도,[Z71],[2.7 가솔린],class_name_examples=Z71 | level_name_examples=...


## Run Category Agent in Batches

한 번에 전부 돌리지 않고 batch 단위로 호출한다.
각 batch가 끝날 때마다 CSV를 저장하므로 오래 걸려도 중간 진행분을 유지할 수 있다.


In [9]:
RUN_AGENT = True
RUN_LIMIT = None  # 예: 40
BATCH_SIZE = 20

if not is_agent_env_ready():
    print('OPENROUTER_API_KEY is missing. Fill cohort_generation/.env first.')
elif not RUN_AGENT:
    print('Set RUN_AGENT = True to execute the category mapping agent.')
else:
    config = CohortAgentConfig.from_env()
    rows_to_run = pending_requests_df.copy()
    if RUN_LIMIT is not None:
        rows_to_run = rows_to_run.head(RUN_LIMIT).copy()

    total_rows = len(rows_to_run)
    appended_batches = []

    for batch_index, start in enumerate(range(0, total_rows, BATCH_SIZE), start=1):
        batch_df = rows_to_run.iloc[start : start + BATCH_SIZE].copy()
        batch_rows = []
        print(f"batch {batch_index}: rows {start + 1}-{start + len(batch_df)} / {total_rows}")

        for request in dataframe_to_mapping_inputs(batch_df):
            result = run_category_mapping_sync(request, config)
            batch_rows.append(result.model_dump(mode="json"))

        if batch_rows:
            batch_result_df = pd.DataFrame(batch_rows)
            appended_batches.append(batch_result_df)
            mapping_table_df = pd.concat([mapping_table_df, batch_result_df], ignore_index=True)
            mapping_table_df = mapping_table_df.drop_duplicates(subset=["brand", "model_name"], keep="last")
            mapping_table_df.to_csv(mapping_table_path, index=False)
            print(f"saved mapping rows: {len(mapping_table_df)}")

    if appended_batches:
        appended_df = pd.concat(appended_batches, ignore_index=True)
        display(appended_df)

    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(mapping_table_df)

    mapping_table_path


debug: model name : openai/gpt-5.2-chat
batch 1: rows 1-20 / 261
saved mapping rows: 26
batch 2: rows 21-40 / 261
saved mapping rows: 46
batch 3: rows 41-60 / 261
saved mapping rows: 66
batch 4: rows 61-80 / 261
saved mapping rows: 86
batch 5: rows 81-100 / 261
saved mapping rows: 106
batch 6: rows 101-120 / 261
saved mapping rows: 126
batch 7: rows 121-140 / 261
saved mapping rows: 146
batch 8: rows 141-160 / 261
saved mapping rows: 166
batch 9: rows 161-180 / 261
saved mapping rows: 186
batch 10: rows 181-200 / 261
saved mapping rows: 206
batch 11: rows 201-220 / 261
saved mapping rows: 226
batch 12: rows 221-240 / 261
saved mapping rows: 246
batch 13: rows 241-260 / 261
saved mapping rows: 266
batch 14: rows 261-261 / 261
saved mapping rows: 267


,brand,model_name,major_category,search_used,note
0,chevrolet,더 뉴 트레일블레이저,suv,False,Chevrolet Trailblazer is sold in Korea as a co...
1,chevrolet,리얼 뉴 콜로라도,truck,False,쉐보레 콜로라도는 중형 픽업트럭 모델
2,chevrolet,말리부,sedan,False,Chevrolet Malibu is a midsize 4-door sedan.
3,chevrolet,볼트 (VOLT),hatchback,False,Chevrolet Volt (EREV) is a 5-door compact hatc...
4,chevrolet,스파크,hatchback,False,Chevrolet Spark is a subcompact 5-door hatchba...
...,...,...,...,...,...
256,renault,마스터,van,False,"르노 마스터는 대형 상용 밴/미니버스 기반 모델로 13·15인승, 밴, 캠핑카 등 ..."
257,renault,올 뉴 SM7,sedan,False,None
258,renault,캡처,suv,False,르노 캡처는 B세그먼트 소형 SUV(크로스오버)로 공식 판매됨
259,renault,클리오,hatchback,False,르노 클리오는 전 세계적으로 판매된 소형 5도어 해치백 모델


,brand,model_name,major_category,search_used,note
0,chevrolet,더 넥스트 스파크,hatchback,False,"Derived from Chevrolet Spark, a subcompact hat..."
1,chevrolet,더 뉴 말리부,sedan,False,NaN
2,chevrolet,더 뉴 스파크,hatchback,True,The user wants mapping only. We have answer.
3,chevrolet,더 뉴 아베오,hatchback,False,"Based on class trims (L, LS, LT) and level nam..."
4,chevrolet,더 뉴 카마로 SS,coupe_convertible,False,"Chevrolet Camaro SS is a 2-door coupe, categor..."
5,chevrolet,더 뉴 트랙스,suv,False,NaN
6,chevrolet,더 뉴 트레일블레이저,suv,False,Chevrolet Trailblazer is sold in Korea as a co...
7,chevrolet,리얼 뉴 콜로라도,truck,False,쉐보레 콜로라도는 중형 픽업트럭 모델
8,chevrolet,말리부,sedan,False,Chevrolet Malibu is a midsize 4-door sedan.
9,chevrolet,볼트 (VOLT),hatchback,False,Chevrolet Volt (EREV) is a 5-door compact hatc...


## Apply Mapping Table to Cohort Input

완성된 mapping table을 `cohort_grouping_input.csv`에 붙여서 category가 포함된 최종 파일을 만든다.


In [10]:
mapping_table_df = load_mapping_table(mapping_table_path)
cohort_grouping_with_category = apply_mapping_table(df, mapping_table_df)
output_path = DATA_PATH.parent / "cohort_grouping_with_category.csv"
cohort_grouping_with_category.to_csv(output_path, index=False)

print("mapped rows:", cohort_grouping_with_category[["major_category"]].dropna().shape[0])
display(cohort_grouping_with_category.head(30))
output_path


mapped rows: 5909


,brand,model_name,level_name,class_name,launch_price_krw,displacement_cc,body_length_mm,body_width_mm,body_height_mm,major_category
0,chevrolet,더 넥스트 스파크,1.0,LS,10360000,999,3595,1595,1475,hatchback
1,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475,hatchback
2,chevrolet,더 넥스트 스파크,1.0,LS 베이직,9990000,999,3595,1595,1475,hatchback
3,chevrolet,더 넥스트 스파크,1.0,LT,11340000,999,3595,1595,1475,hatchback
4,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,995,3595,1595,1475,hatchback
5,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475,hatchback
6,chevrolet,더 넥스트 스파크,1.0,LT 플러스,10360000,999,3595,1595,1475,hatchback
7,chevrolet,더 넥스트 스파크,1.0,LTZ,11360000,995,3595,1595,1475,hatchback
8,chevrolet,더 넥스트 스파크,1.0,LTZ,11340000,999,3595,1595,1475,hatchback
9,chevrolet,더 넥스트 스파크,1.0,LTZ,11340000,999,3595,1595,1475,hatchback


PosixPath('/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/raw/models_cohort/cohort_generation/data/cohort_grouping_with_category.csv')